In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
try:
    from sklearn.model_selection import StratifiedKFold
except ImportError:
    StratifiedKFold = object


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- train_iloc_filter ---
# 10 rows: 5 per class so StratifiedKFold(n_splits=5) works
_X10 = {"x1":[1.,2.,3.,4.,5.,6.,7.,8.,9.,10.],"x2":[11.,12.,13.,14.,15.,16.,17.,18.,19.,20.]}
_y10 = {"y":[0,0,0,0,0,1,1,1,1,1]}
_w10 = {"w":[1.,1.,1.,1.,1.,1.,1.,1.,1.,1.]}
def _make_train_dataset(df, x_columns=None, y_columns=None, w_columns=None, **kw):
    if isinstance(df, pl.DataFrame):
        select = lambda columns: df.select(columns) if columns else df
        first = lambda: df.select(df.columns[:1])
    else:
        select = lambda columns: df[columns] if columns else df
        first = lambda: df.iloc[:, :1]
    return SimpleNamespace(
        df=df,
        X=select(x_columns),
        y=select(y_columns) if y_columns else first(),
        w=select(w_columns) if w_columns else first(),
        x_columns=x_columns, y_columns=y_columns, w_columns=w_columns,
    )

FIX_TRAIN_ILOC_FILTER_CDS = SimpleNamespace(Dataset=_make_train_dataset)
FIX_TRAIN_ILOC_FILTER_DS = SimpleNamespace(
    X=pd.DataFrame(_X10), y=pd.DataFrame(_y10), w=pd.DataFrame(_w10),
    x_columns=["x1","x2"], y_columns=["y"], w_columns=["w"],
    to_pandas=lambda: pd.DataFrame({**_X10, **_y10, **_w10}),
    to_polars=lambda: pl.DataFrame({**_X10, **_y10, **_w10}),
    filter=lambda mask: SimpleNamespace(
        X=pd.DataFrame({"x1":[1.,2.],"x2":[11.,12.]}),
        y=pd.DataFrame({"y":[0,1]}), w=pd.DataFrame({"w":[1.,1.]}),
        x_columns=["x1","x2"], y_columns=["y"], w_columns=["w"],
        to_pandas=lambda: pd.DataFrame({"x1":[1.,2.],"x2":[11.,12.],"y":[0,1],"w":[1.,1.]}),
        to_polars=lambda: pl.DataFrame({"x1":[1.,2.],"x2":[11.,12.],"y":[0,1],"w":[1.,1.]}),
        save=lambda p: None),
    save=lambda p: None)
FIX_TRAIN_ILOC_FILTER_PRED = np.array([0.3,0.7])  # 2 elements — one per test fold row (5-fold on 10 samples)

# --- train_merge_concat ---
def make_train_merge_concat_base_dfs_pd():
    return [pd.DataFrame({"y":[0,1,0],"w":[1.,1.,1.],"pred":[0.3,0.7,0.4]}),
            pd.DataFrame({"y":[1,0],"w":[1.,1.],"pred":[0.8,0.5]})]

def make_train_merge_concat_base_dfs_pl():
    # gen_ resets index → appends DFs with ["index","y","w","pred"] (4 cols)
    return [pl.DataFrame({"index":[0,1,2],"y":[0,1,0],"w":[1.,1.,1.],"pred":[0.3,0.7,0.4]}),
            pl.DataFrame({"index":[3,4],"y":[1,0],"w":[1.,1.],"pred":[0.8,0.5]})]

FIX_TRAIN_MERGE_CONCAT_PRED = np.array([0.3, 0.7, 0.4, 0.8, 0.5])
FIX_TRAIN_MERGE_CONCAT_TEST_DS = SimpleNamespace(
    X=pd.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.]}),
    y=pd.DataFrame({"y":[0,1,0,1,0]}), w=pd.DataFrame({"w":[1.,1.,1.,1.,1.]}),
    x_columns=["x1","x2"], y_columns=["y"], w_columns=["w"],
    to_pandas=lambda: pd.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.],"y":[0,1,0,1,0],"w":[1.,1.,1.,1.,1.]}),
    to_polars=lambda: pl.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.],"y":[0,1,0,1,0],"w":[1.,1.,1.,1.,1.]}),
    filter=lambda mask: SimpleNamespace(
        X=pd.DataFrame({"x1":[1.,2.],"x2":[4.,5.]}),
        y=pd.DataFrame({"y":[0,1]}), w=pd.DataFrame({"w":[1.,1.]}),
        x_columns=["x1","x2"], y_columns=["y"], w_columns=["w"],
        to_pandas=lambda: pd.DataFrame({"x1":[1.,2.],"x2":[4.,5.],"y":[0,1],"w":[1.,1.]}),
        to_polars=lambda: pl.DataFrame({"x1":[1.,2.],"x2":[4.,5.],"y":[0,1],"w":[1.,1.]}),
        save=lambda p: None),
    save=lambda p: None)

print("✅ Fixtures loaded")

In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_train_iloc_filter(cds, ds, pred):
    base_dfs = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for epoch, (train_index, test_index) in enumerate(skf.split(ds.X, ds.y)):
        _train_ds = cds.Dataset(
            df=ds.to_pandas().iloc[train_index],
            x_columns=ds.x_columns,
            y_columns=ds.y_columns,
            w_columns=ds.w_columns,
        )
        test_ds = cds.Dataset(
            df=ds.to_pandas().iloc[test_index],
            x_columns=ds.x_columns,
            y_columns=ds.y_columns,
            w_columns=ds.w_columns,
        )
        _pred_df = pd.DataFrame(
            {"index": test_ds.y.index, "pred": pred.reshape(-1)}
        ).set_index("index")
        _base_df = pd.merge(
            test_ds.y.rename(columns={test_ds.y_columns[0]: "y"}),
            test_ds.w.rename(columns={test_ds.w_columns[0]: "w"}),
            left_index=True,
            right_index=True,
        )
        _base_df = pd.merge(_base_df, _pred_df, left_index=True, right_index=True)
        base_dfs.append(_base_df)
    base_df = pd.concat(base_dfs)
    return base_df

def before_train_merge_concat(base_dfs, pred, test_ds):
    _pred_df = pd.DataFrame(
        {"index": test_ds.y.index, "pred": pred.reshape(-1)}
    ).set_index("index")
    _base_df = pd.merge(
        test_ds.y.rename(columns={test_ds.y_columns[0]: "y"}),
        test_ds.w.rename(columns={test_ds.w_columns[0]: "w"}),
        left_index=True,
        right_index=True,
    )
    _base_df = pd.merge(
        _base_df,
        _pred_df,
        left_index=True,
        right_index=True,
    )
    base_dfs.append(_base_df)
    base_df = pd.concat(base_dfs)
    return base_df

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_train_iloc_filter(cds, ds, pred):

    base_dfs = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    _full_df = ds.to_polars().with_row_index("index")
    for epoch, (train_index, test_index) in enumerate(skf.split(ds.X, ds.y)):
        _train_ds = cds.Dataset(
            df=_full_df[train_index].drop("index"),
            x_columns=ds.x_columns,
            y_columns=ds.y_columns,
            w_columns=ds.w_columns,
        )
        test_df = _full_df[test_index]
        test_ds = cds.Dataset(
            df=test_df.drop("index"),
            x_columns=ds.x_columns,
            y_columns=ds.y_columns,
            w_columns=ds.w_columns,
        )
        _pred_df = pl.DataFrame(
            {"index": test_df["index"], "pred": pred.reshape(-1)}
        )
        _base_df = test_df.select(
            pl.col(test_ds.y_columns[0]).alias("y"),
            pl.col(test_ds.w_columns[0]).alias("w"),
            pl.col("index"),
        )
        _base_df = _base_df.join(_pred_df, on="index", how="inner")
        base_dfs.append(_base_df)
    base_df = pl.concat(base_dfs)
    return base_df

def gen_train_merge_concat(base_dfs, pred, test_ds):
    _generated_code = 'import polars as pl\n\n        _pred_df = pl.DataFrame(\n            {"index": test_ds.y.index, "pred": pred.reshape(-1)}\n        )\n        _base_df = pl.DataFrame(\n            {\n                "index": test_ds.y.index,\n                "y": test_ds.y.iloc[:, 0].to_numpy(),\n            }\n        ).join(\n            pl.DataFrame(\n                {\n                    "index": test_ds.w.index,\n                    "w": test_ds.w.iloc[:, 0].to_numpy(),\n                }\n            ),\n            on="index",\n            how="inner",\n        )\n        _base_df = _base_df.join(\n            _pred_df,\n            on="index",\n            how="inner",\n        )\n        base_dfs.append(_base_df)\n    base_df = pl.concat(base_dfs)'
    raise SyntaxError('generated code for train_merge_concat is not syntactically valid after wrapper normalization: mixed/inconsistent indentation could not be safely normalized')

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: train_iloc_filter ===

# L1 smoke – generated
try:
    _r = gen_train_iloc_filter(FIX_TRAIN_ILOC_FILTER_CDS, FIX_TRAIN_ILOC_FILTER_DS, FIX_TRAIN_ILOC_FILTER_PRED)
    print("✅ L1 smoke gen_train_iloc_filter: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_train_iloc_filter: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_train_iloc_filter(FIX_TRAIN_ILOC_FILTER_CDS, FIX_TRAIN_ILOC_FILTER_DS, FIX_TRAIN_ILOC_FILTER_PRED)
    print("✅ L1 smoke before_train_iloc_filter: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_train_iloc_filter: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_train_iloc_filter(FIX_TRAIN_ILOC_FILTER_CDS, FIX_TRAIN_ILOC_FILTER_DS, FIX_TRAIN_ILOC_FILTER_PRED)
    _rg = gen_train_iloc_filter(FIX_TRAIN_ILOC_FILTER_CDS, FIX_TRAIN_ILOC_FILTER_DS, FIX_TRAIN_ILOC_FILTER_PRED)
    compare(_rb, _rg, "train_iloc_filter", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence train_iloc_filter: setup error — {type(_e).__name__}: {_e}")

# L3 edge – zero predictions for every fold row
try:
    _zero_pred = np.zeros_like(FIX_TRAIN_ILOC_FILTER_PRED)
    _rb = before_train_iloc_filter(FIX_TRAIN_ILOC_FILTER_CDS, FIX_TRAIN_ILOC_FILTER_DS, _zero_pred)
    _rg = gen_train_iloc_filter(FIX_TRAIN_ILOC_FILTER_CDS, FIX_TRAIN_ILOC_FILTER_DS, _zero_pred)
    compare(_rb, _rg, "L3 edge train_iloc_filter zero predictions", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge train_iloc_filter zero predictions: {type(_e).__name__}: {_e}")
